# 🗺️ Google Maps Platform API 종합 테스트 노트북

본 노트북은 **Google Maps Platform**의 주요 Web Service API 및 최신 **Places API (New / Modern v1)**의 모든 기능과 반환 데이터를 체계적으로 테스트하고 시각화할 수 있도록 구성되어 있습니다.

---

### 📌 다루는 주요 API 서비스
| 번호 | 서비스명 | 주요 기능 및 반환 정보 | 연동 방식 |
| :---: | :--- | :--- | :---: |
| **01** | **Geocoding API** | 주소 ➡️ 위도/경도 좌표 변환, 정밀도(ROOFTOP 등), 주소 계층 컴포넌트 분해 | Python SDK |
| **02** | **Reverse Geocoding API** | 위도/경도 좌표 ➡️ 표준 주소 및 행정구역 명칭 역조회 | Python SDK |
| **03** | **Places API (New) - 검색** | 텍스트 검색(SearchText) 및 반경 기반 주변 시설 검색(SearchNearby) | REST (v1) |
| **04** | **Places API (New) - 상세** | 와일드카드(`*`) 필드마스크를 통한 기본 정보, 리뷰, 편의시설, 전기차 충전 등 전체 필드 조회 | REST (v1) |
| **05** | **Places Autocomplete API** | 실시간 검색어 자동완성 및 추천 장소 후보군 조회 | REST (v1) |
| **06** | **Directions API** | 출발지-도착지 간 최적 경로, 실시간 교통 반영 소요시간, 턴바이턴 안내 | Python SDK |
| **07** | **Distance Matrix API** | 다중 출발지-도착지 간 이동 거리 및 소요 시간 매트릭스 계산 | Python SDK |
| **08** | **Elevation API** | 특정 좌표 및 지점의 해발 고도 측정 | Python SDK |
| **09** | **Time Zone API** | 지정 좌표 위치의 현지 타임존 ID, UTC 오프셋 및 서머타임(DST) 정보 | Python SDK |
| **10** | **Geolocation API** | 네트워크/기지국/IP 신호 기반 기기 추정 위치 및 정확도 반경 | Python SDK |

---


## 📦 0. 환경 설정 및 패키지 설치 가이드

본 프로젝트는 초고속 패키지 관리자 `uv` 및 표준 `pip` 환경을 모두 지원합니다.

### ⚡ Option A. `uv` 사용 (권장 - 초고속 가상환경 관리)

프로젝트 루트 디렉토리(`/Users/hangsik/Documents/my_project/google_apis`)에서 아래 명령어로 가상환경 구성 및 패키지를 설치합니다:

```bash
# 1. 가상환경 생성 및 의존성 동기화
uv sync

# 2. Jupyter 커널 등록
uv run python -m ipykernel install --user --name google_apis_env --display-name "Python (google_apis_env)"
```

### 🐍 Option B. 표준 `pip` / `venv` 사용

```bash
# 1. 가상환경 생성
python3 -m venv .venv

# 2. 가상환경 활성화 (macOS / Linux)
source .venv/bin/activate

# 3. 필수 패키지 설치
pip install googlemaps requests pandas python-dotenv ipykernel

# 4. Jupyter 커널 등록
python -m ipykernel install --user --name google_apis_env --display-name "Python (google_apis_env)"
```

### 🔑 `.env` 파일 설정

프로젝트 루트의 `.env` 파일에 발급받은 Google Maps API Key를 설정합니다:

```env
GOOGLE_MAPS_API_KEY=AIzaSy...your_actual_api_key_here
```

> [!TIP]
> Google Cloud 콘솔에서 각 API(Geocoding, Places API (New), Directions, Distance Matrix, Elevation, Time Zone, Geolocation)가 활성화되어 있는지 확인하세요.


In [ ]:
import os
import json
import datetime
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv
import googlemaps

# 루트 디렉토리의 .env 파일을 자동으로 탐색하고 로드 (override=True)
env_path = find_dotenv()
load_dotenv(env_path, override=True)

# 깔끔한 JSON 출력을 위한 헬퍼 함수
def print_json(data, title=None, max_lines=40):
    if title:
        print(f"\n=== {title} ===")
    formatted = json.dumps(data, indent=2, ensure_ascii=False, default=str)
    lines = formatted.splitlines()
    if len(lines) > max_lines:
        print("\n".join(lines[:max_lines]))
        print(f"... (총 {len(lines)}줄 중 {max_lines}줄 출력됨 - 전체 내용은 변수 참조)")
    else:
        print(formatted)

print(f"✅ .env 파일 로드 완료: {env_path if env_path else '기본 환경변수 사용'}")


## 🔑 1. API 키 로드 및 클라이언트 초기화

`.env` 파일에서 `GOOGLE_MAPS_API_KEY`를 안전하게 로드하고 공식 `googlemaps.Client` 인스턴스를 초기화합니다.


In [ ]:
# 환경 변수에서 API 키 로드
API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")

if API_KEY:
    API_KEY = API_KEY.strip().strip('"').strip("'")

if not API_KEY or API_KEY == "YOUR_GOOGLE_MAPS_API_KEY_HERE":
    import getpass
    API_KEY = getpass.getpass("Google Maps API Key를 입력하세요: ").strip().strip('"').strip("'")

# Google Maps 공식 Python SDK 클라이언트 초기화
try:
    gmaps = googlemaps.Client(key=API_KEY)
    masked_key = f"{API_KEY[:6]}...{API_KEY[-4:]}" if len(API_KEY) > 10 else "***"
    print(f"✅ Google Maps 클라이언트 초기화 성공 (키: {masked_key})")
except Exception as e:
    print("❌ 클라이언트 초기화 실패:", e)


## 📍 2. Geocoding API (주소 ➡️ 좌표 지오코딩)

주소 문자열을 위도/경도 좌표, 정규화된 표준 주소, 행정 구역 계층 구성요소로 변환합니다.

### 📌 조회되는 핵심 정보:
- **표준 주소 (`formatted_address`)**: 정규화된 표준 전체 주소
- **좌표 (`location`)**: 위도(`lat`), 경도(`lng`)
- **위치 정밀도 (`location_type`)**: `ROOFTOP`(정확한 건물 위치), `RANGE_INTERPOLATED`, `GEOMETRIC_CENTER`, `APPROXIMATE`
- **주소 구성요소 (`address_components`)**: 도로명, 건물번호, 구/군, 시/도, 국가, 우편번호
- **뷰포트 및 바운딩 박스 (`viewport` / `bounds`)**: 지도 표시용 경계 좌표
- **Place ID**: Google 고유 장소 식별자


In [ ]:
target_address = "1600 Amphitheatre Parkway, Mountain View, CA 94043, USA"

try:
    geocode_result = gmaps.geocode(target_address)
    print(f"주소 '{target_address}' 에 대해 {len(geocode_result)}건의 결과 발견\n")

    if geocode_result:
        first_result = geocode_result[0]
        
        # 1. 기본 및 좌표 정보
        print("📌 [표준 주소]:", first_result.get("formatted_address"))
        print("🆔 [Place ID]:", first_result.get("place_id"))
        print("🏷️ [장소 유형]:", first_result.get("types"))
        
        location = first_result.get("geometry", {}).get("location", {})
        print(f"🌐 [좌표 정보]: 위도(Lat) = {location.get('lat')}, 경도(Lng) = {location.get('lng')}")
        print("🎯 [위치 정밀도]:", first_result.get("geometry", {}).get("location_type"))
        print("📐 [뷰포트 경계]:", first_result.get("geometry", {}).get("viewport"))
        
        # 2. 주소 상세 구성요소 표
        components = first_result.get("address_components", [])
        df_components = pd.DataFrame(components)
        print("\n📋 [주소 계층 상세 구성요소]:")
        display(df_components)
        
        # 3. 원본 JSON 요약 출력
        print_json(first_result, title="Geocoding API 응답 JSON")
except Exception as e:
    print("❌ Geocoding API 오류:", e)


## 🔄 3. Reverse Geocoding API (좌표 ➡️ 주소 역지오코딩)

위도/경도 좌표로부터 일치하는 표준 주소 및 행정 구역 명칭 목록을 조회합니다.


In [ ]:
# 예시: Google 한국 GFC 오피스 좌표 (강남구 테헤란로 152)
coords = (37.50005, 127.0365)

try:
    reverse_results = gmaps.reverse_geocode(coords, language="ko")
    print(f"좌표 {coords}에 대해 {len(reverse_results)}건의 행정구역/주소 정보 조회됨:\n")

    records = []
    for idx, item in enumerate(reverse_results):
        records.append({
            "순번": idx,
            "표준 주소": item.get("formatted_address"),
            "Place ID": item.get("place_id"),
            "정밀도": item.get("geometry", {}).get("location_type"),
            "유형": ", ".join(item.get("types", []))
        })

    df_reverse = pd.DataFrame(records)
    display(df_reverse)

    if reverse_results:
        print_json(reverse_results[0], title="최상위 역지오코딩 매칭 결과 (JSON)")
except Exception as e:
    print("❌ Reverse Geocoding API 오류:", e)


## 🔍 4. Places API (New): 장소 텍스트 검색 및 주변 검색

최신 **Places API Modern v1** 엔드포인트를 호출하여 텍스트 쿼리 기반 검색 및 특정 위치 반경 내 주변 시설을 검색합니다.

- **텍스트 검색 (`places:searchText`)**: `"Googleplex Mountain View"` 등 자연어 장소 검색
- **주변 시설 검색 (`places:searchNearby`)**: 특정 좌표 중심 반경(Circle) 내 카테고리별(카페, 식당 등) 검색


In [ ]:
# 4.1 Places API (New) - 텍스트 기반 장소 검색
url = "https://places.googleapis.com/v1/places:searchText"
headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.rating,places.userRatingCount,places.location,places.types,places.regularOpeningHours,places.businessStatus"
}

body = {
    "textQuery": "Googleplex Mountain View",
    "languageCode": "ko"  # 한국어 응답 요청
}

sample_place_id = None
response = requests.post(url, headers=headers, json=body)

if response.status_code == 200:
    data = response.json()
    places = data.get("places", [])
    print(f"✅ 텍스트 검색 결과 {len(places)}건 반환됨:\n")
    
    search_summary = []
    for p in places:
        search_summary.append({
            "장소명": p.get("displayName", {}).get("text"),
            "주소": p.get("formattedAddress"),
            "평점": p.get("rating"),
            "리뷰 수": p.get("userRatingCount"),
            "영업 상태": p.get("businessStatus"),
            "Place ID": p.get("id")
        })
    
    df_search = pd.DataFrame(search_summary)
    display(df_search)
    
    if places:
        sample_place_id = places[0].get("id")
        print(f"🎯 상세 조회용 Place ID 선택: {sample_place_id}")
    
    print_json(data, title="Places API (New) 텍스트 검색 응답")
else:
    print(f"❌ Places API (New) 오류 ({response.status_code}):", response.text)


In [ ]:
# 4.2 Places API (New) - 주변 시설 검색 (예: 1.5km 반경 내 카페/음식점)
url = "https://places.googleapis.com/v1/places:searchNearby"
headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.rating,places.userRatingCount,places.primaryType,places.location"
}

body = {
    "includedTypes": ["cafe", "coffee_shop", "restaurant"],
    "maxResultCount": 10,
    "locationRestriction": {
        "circle": {
            "center": {
                "latitude": 37.422388,
                "longitude": -122.0841883
            },
            "radius": 1500.0  # 반경 1,500 미터
        }
    }
}

response = requests.post(url, headers=headers, json=body)

if response.status_code == 200:
    nearby_data = response.json()
    nearby_places = nearby_data.get("places", [])
    print(f"✅ 주변 1.5km 내 {len(nearby_places)}개 시설 발견:\n")
    
    nearby_list = []
    for p in nearby_places:
        nearby_list.append({
            "장소명": p.get("displayName", {}).get("text"),
            "기본 유형": p.get("primaryType"),
            "평점": p.get("rating"),
            "리뷰 수": p.get("userRatingCount"),
            "주소": p.get("formattedAddress"),
            "Place ID": p.get("id")
        })
        
    df_nearby = pd.DataFrame(nearby_list)
    display(df_nearby)
else:
    print(f"❌ Places 주변 검색 오류 ({response.status_code}):", response.text)


## 🏢 5. Places API (New): 장소 상세 정보 전체 필드 조회 (`*` Wildcard FieldMask)

와일드카드 필드마스크 `X-Goog-FieldMask: *`를 사용하여 장소에 대해 Google이 제공하는 **모든 정보**를 완전히 추출합니다.

### 📌 조회되는 전체 정보 카테고리:
- **기본 식별 및 지오메트리**: `id`, `displayName`, `formattedAddress`, `location`, `viewport`, `plusCode`, `googleMapsUri`, `types`
- **평가 및 분위기**: `rating`, `userRatingCount`, `priceLevel`, `editorialSummary`, `reviews`
- **연락처 및 웹사이트**: `nationalPhoneNumber`, `internationalPhoneNumber`, `websiteUri`
- **영업시간**: `regularOpeningHours`, `currentOpeningHours`, 요일별 세부 스케줄
- **현대적 편의시설 및 옵션**: `accessibilityOptions`(접근성), `parkingOptions`(주차), `paymentOptions`(결제수단), `evChargeOptions`(전기차 충전), `dineIn`, `delivery`, `takeout`, `servesVegetarianFood`
- **사진 정보**: `photos` (사진 레퍼런스, 규격, 저작자 표시)


In [ ]:
# 장소 상세 정보 전체 조회
target_place_id = sample_place_id or "ChIJj61dQgK6j4AR4GeTYWZsKWw"

url = f"https://places.googleapis.com/v1/places/{target_place_id}"
headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "*"  # 모든 필드 요청
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    full_place_data = response.json()
    print("✅ 장소 상세 정보 전체 페이로드 수신 완료!")
    print(f"제공된 최상위 필드 수: {len(full_place_data.keys())}개\n")
    
    # 1. 기본 식별 및 위치
    print("=== 📌 1. 기본 식별 및 위치 ===")
    print("장소명:", full_place_data.get("displayName", {}).get("text"))
    print("Place ID:", full_place_data.get("id"))
    print("주소:", full_place_data.get("formattedAddress"))
    print("좌표:", full_place_data.get("location"))
    print("지도 URL:", full_place_data.get("googleMapsUri"))
    print("카테고리 유형:", full_place_data.get("types"))
    
    # 2. 연락처 및 웹 정보
    print("\n=== 📞 2. 연락처 및 웹 정보 ===")
    print("국내 전화번호:", full_place_data.get("nationalPhoneNumber"))
    print("국제 전화번호:", full_place_data.get("internationalPhoneNumber"))
    print("웹사이트:", full_place_data.get("websiteUri"))
    
    # 3. 평가 및 요약
    print("\n=== ⭐ 3. 평가 및 소개 요약 ===")
    print("평점:", full_place_data.get("rating"))
    print("총 평가 수:", full_place_data.get("userRatingCount"))
    print("가격대 레벨:", full_place_data.get("priceLevel"))
    print("소개 요약(Editorial):", full_place_data.get("editorialSummary", {}).get("text"))
    
    # 4. 편의시설 및 서비스 옵션
    print("\n=== 🍽️ 4. 편의시설 및 서비스 옵션 ===")
    service_options = {
        "매장 내 식사 (Dine In)": full_place_data.get("dineIn"),
        "배달 (Delivery)": full_place_data.get("delivery"),
        "포장 (Takeout)": full_place_data.get("takeout"),
        "커브사이드 픽업": full_place_data.get("curbsidePickup"),
        "예약 가능 여부": full_place_data.get("reservable"),
        "맥주 제공": full_place_data.get("servesBeer"),
        "와인 제공": full_place_data.get("servesWine"),
        "채식 메뉴 제공": full_place_data.get("servesVegetarianFood"),
        "화장실 구비": full_place_data.get("restroom"),
        "접근성 옵션 (휠체어 등)": full_place_data.get("accessibilityOptions"),
        "주차 옵션": full_place_data.get("parkingOptions"),
        "결제 수단 옵션": full_place_data.get("paymentOptions"),
        "전기차 충전 옵션 (EV)": full_place_data.get("evChargeOptions")
    }
    for k, v in service_options.items():
        if v is not None:
            print(f"- {k}: {v}")
            
    # 5. 고객 리뷰 표
    reviews = full_place_data.get("reviews", [])
    print(f"\n=== 💬 5. 고객 리뷰 ({len(reviews)}건 제공) ===")
    review_list = []
    for r in reviews:
        review_list.append({
            "작성자": r.get("authorAttribution", {}).get("displayName"),
            "평점": r.get("rating"),
            "작성 시간": r.get("relativePublishTimeDescription"),
            "리뷰 내용": (r.get("text", {}).get("text", "")[:120] + "...") if len(r.get("text", {}).get("text", "")) > 120 else r.get("text", {}).get("text", "")
        })
    df_reviews = pd.DataFrame(review_list)
    display(df_reviews)
    
    # 6. 전체 JSON 원본 요약 출력
    print_json(full_place_data, title="장소 상세 정보 원본 JSON (*)")
else:
    print(f"❌ 오류 발생 ({response.status_code}):", response.text)


## ✍️ 6. Places Autocomplete API (검색어 자동완성)

사용자 입력 텍스트에 따른 실시간 장소명, 주소 자동완성 추천 결과 후보군을 제공합니다.


In [ ]:
url = "https://places.googleapis.com/v1/places:autocomplete"
headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY
}

body = {
    "input": "Golden Gate",
    "includedPrimaryTypes": ["tourist_attraction", "park", "establishment"]
}

response = requests.post(url, headers=headers, json=body)

if response.status_code == 200:
    auto_data = response.json()
    suggestions = auto_data.get("suggestions", [])
    print(f"✅ 자동완성 결과 {len(suggestions)}건 반환:\n")
    
    sugg_list = []
    for s in suggestions:
        pred = s.get("placePrediction", {})
        sugg_list.append({
            "추천 텍스트": pred.get("text", {}).get("text"),
            "Place ID": pred.get("placeId"),
            "유형": ", ".join(pred.get("types", []))
        })
    df_sugg = pd.DataFrame(sugg_list)
    display(df_sugg)
else:
    print(f"❌ 자동완성 API 오류 ({response.status_code}):", response.text)


## 🚗 7. Directions API (경로 탐색, 턴바이턴 내비게이션 & 교통 정보)

출발지와 목적지 간 이동 경로, 대안 경로(Alternatives), 실시간 교통 반영 소요시간, 단계별 회전 안내(Turn-by-turn Navigation)를 제공합니다.


In [ ]:
origin = "San Francisco, CA"
destination = "Mountain View, CA"

try:
    directions_result = gmaps.directions(
        origin=origin,
        destination=destination,
        mode="driving",
        departure_time="now",
        traffic_model="best_guess",
        alternatives=True
    )

    print(f"✅ '{origin}' ➡️ '{destination}' 간 {len(directions_result)}개 경로 발견:\n")

    for route_idx, route in enumerate(directions_result):
        summary = route.get("summary")
        leg = route.get("legs", [{}])[0]
        distance = leg.get("distance", {}).get("text")
        duration = leg.get("duration", {}).get("text")
        duration_in_traffic = leg.get("duration_in_traffic", {}).get("text", "정보 없음")

        print(f"🛣️ 경로 #{route_idx + 1}: 주요 경유 ({summary})")
        print(f"   총 거리: {distance}")
        print(f"   표준 소요시간: {duration}")
        print(f"   실시간 교통 반영 소요시간: {duration_in_traffic}\n")

    # 기본 추천 경로의 단계별 내비게이션 안내
    if directions_result:
        primary_route = directions_result[0]
        steps = primary_route["legs"][0]["steps"]
        print(f"📋 단계별 턴바이턴 경로 안내 ({len(steps)}개 단계 중 상위 10개 표시):")

        step_records = []
        for i, step in enumerate(steps):
            import re
            clean_instruction = re.sub('<[^<]+?>', '', step.get("html_instructions", ""))
            step_records.append({
                "단계": i + 1,
                "주행 안내": clean_instruction,
                "구간 거리": step.get("distance", {}).get("text"),
                "구간 소요시간": step.get("duration", {}).get("text"),
                "동작 (Maneuver)": step.get("maneuver", "직진")
            })

        df_steps = pd.DataFrame(step_records)
        display(df_steps.head(10))
except Exception as e:
    print("❌ Directions API 오류:", e)


## 📏 8. Distance Matrix API (다중 출발지-목적지 거리 행렬)

여러 출발지와 목적지 간의 이동 거리 및 소요 시간을 매트릭스 형태로 일괄 계산합니다.


In [ ]:
origins = ["San Francisco, CA", "Oakland, CA", "San Jose, CA"]
destinations = ["Mountain View, CA", "Palo Alto, CA"]

try:
    matrix_result = gmaps.distance_matrix(
        origins=origins,
        destinations=destinations,
        mode="driving",
        departure_time="now"
    )

    matrix_rows = []
    for i, origin_name in enumerate(matrix_result.get("origin_addresses", [])):
        row_elements = matrix_result["rows"][i]["elements"]
        for j, dest_name in enumerate(matrix_result.get("destination_addresses", [])):
            element = row_elements[j]
            if element.get("status") == "OK":
                matrix_rows.append({
                    "출발지": origin_name,
                    "도착지": dest_name,
                    "이동 거리": element.get("distance", {}).get("text"),
                    "표준 소요시간": element.get("duration", {}).get("text"),
                    "교통 반영 소요시간": element.get("duration_in_traffic", {}).get("text", "정보 없음")
                })

    df_matrix = pd.DataFrame(matrix_rows)
    display(df_matrix)
except Exception as e:
    print("⚠️ Distance Matrix API 참고:", e)
    print("💡 미활성화된 경우 Google Cloud 콘솔에서 활성화 필요: https://console.cloud.google.com/apis/library/distancematrix-backend.googleapis.com")


## ⛰️ 9. Elevation API (고도 측정)

특정 좌표 지점의 해발 고도 및 경로를 따른 고도 변화 프로파일을 조회합니다.


In [ ]:
landmarks = [
    {"name": "에베레스트 정상", "coords": (27.9881, 86.9250)},
    {"name": "데스밸리 (배드워터)", "coords": (36.2503, -116.8258)},
    {"name": "구글 본사 (마운틴뷰)", "coords": (37.4220, -122.0841)}
]

coords_list = [l["coords"] for l in landmarks]

try:
    elevation_results = gmaps.elevation(coords_list)
    elevation_records = []
    for l, res in zip(landmarks, elevation_results):
        elevation_records.append({
            "위치 명칭": l["name"],
            "위도": res.get("location", {}).get("lat"),
            "경도": res.get("location", {}).get("lng"),
            "고도 (미터)": f"{res.get('elevation'):.2f} m",
            "고도 (피트)": f"{res.get('elevation') * 3.28084:.2f} ft",
            "측정 해상도": f"{res.get('resolution'):.2f} m"
        })
    df_elevation = pd.DataFrame(elevation_records)
    display(df_elevation)
except Exception as e:
    print("⚠️ Elevation API 참고:", e)
    print("💡 미활성화된 경우 Google Cloud 콘솔에서 활성화 필요: https://console.cloud.google.com/apis/library/elevation-backend.googleapis.com")


## ⏰ 10. Time Zone API (시간대 조회)

지정된 좌표 위치의 현지 타임존 ID, 명칭, 표준 UTC 오프셋 및 서머타임(DST) 정보를 계산합니다.


In [ ]:
cities = [
    {"city": "대한민국 서울", "coords": (37.5665, 126.9780)},
    {"city": "미국 뉴욕", "coords": (40.7128, -74.0060)},
    {"city": "영국 런던", "coords": (51.5074, -0.1278)}
]

now_timestamp = datetime.datetime.now(datetime.timezone.utc).timestamp()

try:
    timezone_records = []
    for c in cities:
        tz_res = gmaps.timezone(location=c["coords"], timestamp=now_timestamp)
        if tz_res.get("status") == "OK":
            raw_offset_hours = tz_res.get("rawOffset", 0) / 3600
            dst_offset_hours = tz_res.get("dstOffset", 0) / 3600
            total_offset = raw_offset_hours + dst_offset_hours
            timezone_records.append({
                "도시": c["city"],
                "타임존 ID": tz_res.get("timeZoneId"),
                "타임존 명칭": tz_res.get("timeZoneName"),
                "표준 오프셋": f"{raw_offset_hours:+.1f} 시간",
                "서머타임(DST)": f"{dst_offset_hours:+.1f} 시간",
                "총 UTC 오프셋": f"UTC{total_offset:+.1f}"
            })
    df_tz = pd.DataFrame(timezone_records)
    display(df_tz)
except Exception as e:
    print("⚠️ Time Zone API 참고:", e)
    print("💡 미활성화된 경우 Google Cloud 콘솔에서 활성화 필요: https://console.cloud.google.com/apis/library/timezone-backend.googleapis.com")


## 📶 11. Geolocation API (네트워크 기반 위치 추정)

Wi-Fi 및 네트워크 게이트웨이 신호 정보를 기반으로 현재 기기의 추정 좌표와 정확도 반경을 반환합니다.


In [ ]:
try:
    geolocate_res = gmaps.geolocate(consider_ip=True)
    print("✅ [추정 기기/게이트웨이 위치 정보]:")
    print("추정 좌표:", geolocate_res.get("location"))
    print("정확도 반경 (미터):", geolocate_res.get("accuracy"))
    print_json(geolocate_res, title="Geolocation API 응답")
except Exception as e:
    print("⚠️ Geolocation API 참고:", e)
    print("💡 미활성화된 경우 Google Cloud 콘솔에서 활성화 필요: https://console.cloud.google.com/apis/library/geolocation.googleapis.com")


## 📊 12. 조회된 정보 요약표

| API 서비스 | 연동 방식 | 주요 추출 및 조회 정보 |
| :--- | :---: | :--- |
| **Geocoding (지오코딩)** | Python SDK | 위도/경도 좌표, 세부 행정구역 계층, 표준 주소, 뷰포트, 바운딩 박스, 정밀도 타입, Place ID |
| **Reverse Geocoding (역지오코딩)** | Python SDK | 좌표 기반 표준 주소, 행정 경계 매핑 |
| **Places API New (장소 검색)** | REST v1 | 장소명, 평점, 리뷰 수, 표준 주소, 좌표, 영업 상태, 기본 카테고리 |
| **Places API New (장소 상세 `*`)** | REST v1 | 전체 필드마스크: 소개 요약(Editorial), 고객 리뷰, 사진, 영업시간, 편의시설, 접근성(휠체어), 주차, 전기차 충전(EV), 결제수단 |
| **Places Autocomplete (자동완성)** | REST v1 | 실시간 자동완성 추천어, 추천 장소 ID, 카테고리 |
| **Directions (경로 탐색)** | Python SDK | 턴바이턴 단계별 주행 안내, 실시간 교통 반영 소요시간, 인코딩 폴리라인, 대안 경로 |
| **Distance Matrix (거리 행렬)** | Python SDK | 다중 출발지-도착지 간 이동 거리 및 소요 시간 매트릭스 |
| **Elevation (고도 측정)** | Python SDK | 해발 고도, 측정 해상도 |
| **Time Zone (시간대)** | Python SDK | 타임존 ID, 타임존 명칭, 표준 UTC 오프셋, 서머타임 오프셋 |
| **Geolocation (지오로케이션)** | Python SDK | 네트워크/IP 기반 추정 좌표 및 정확도 반경 |

---

### 📚 공식 문서 및 레퍼런스
- [Google Maps Platform 문서](https://developers.google.com/maps/documentation)
- [Places API (New) 문서](https://developers.google.com/maps/documentation/places/web-service/op-overview)
- [google-maps-services-python GitHub](https://github.com/googlemaps/google-maps-services-python)
